In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [2]:
results_1l = pd.read_excel("resultados-1l.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
    # [results_1l, results_2l, results_3l],
    # ignore_index=True
# )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_theta,R2diff_ZZx1_theta,R2_ZZx2_theta,R2diff_ZZx2_theta,...,R2_LSG_1_theta,R2diff_LSG_1_theta,R2_LSG_2_theta,R2diff_LSG_2_theta,R2_ZZx1_inv_theta,R2diff_ZZx1_inv_theta,R2_zzx2_inv2_theta,R2diff_zzx2_inv2_theta,R2_semiCirc_theta,R2diff_semiCirc_theta
0,model_arch1_r0.01_Ld0.5_Lp0.5_seed4556,[1],0.5,0.5,0.01,4556,-0.167582,0.000039,0.027286,-0.001014,...,-12.284592,-0.056109,-0.735623,-0.010584,-1.579825,-0.025811,-0.957681,-0.002717,-0.056048,-0.009349
1,model_arch1_r0.01_Ld0.5_Lp0.5_seed6942,[1],0.5,0.5,0.01,6942,0.155277,0.496816,-0.317307,0.335895,...,0.803915,0.554479,-2.393724,0.331857,-0.782529,0.216842,-7.187892,-0.027522,-29.654581,-0.211164
2,model_arch1_r0.01_Ld0.5_Lp0.5_seed7320,[1],0.5,0.5,0.01,7320,0.327859,0.526272,0.113966,0.356302,...,-1.001378,0.467437,-3.033300,0.259641,-0.642934,0.256139,-7.042551,0.104355,-14.594577,0.031130
3,model_arch1_r0.01_Ld0.5_Lp0.5_seed1896,[1],0.5,0.5,0.01,1896,0.262544,0.591508,0.390693,0.407808,...,0.657994,0.574531,-4.236300,0.296886,-1.047151,0.233023,-9.216149,-0.020344,-32.543729,-0.260557
4,model_arch1_r0.01_Ld0.5_Lp0.5_seed1628,[1],0.5,0.5,0.01,1628,0.573711,0.491707,-0.251973,0.329404,...,0.745974,0.527540,-3.974377,0.280791,-0.332402,0.162216,-6.996865,-0.029852,-28.444689,-0.204702
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,model_arch100_r0.9_Ld0.7_Lp0.3_seed4556,[100],0.7,0.3,0.90,4556,0.493461,0.620887,0.660107,0.465532,...,0.429331,0.562024,-1.860737,0.337967,-1.193717,0.429834,-10.385476,0.114966,-23.346503,-0.078027
2996,model_arch100_r0.9_Ld0.7_Lp0.3_seed6942,[100],0.7,0.3,0.90,6942,0.506604,0.639326,0.528228,0.439090,...,0.862826,0.603856,-1.697028,0.370111,-0.984529,0.453153,-11.762284,-0.021217,-33.794879,-0.265492
2997,model_arch100_r0.9_Ld0.7_Lp0.3_seed7320,[100],0.7,0.3,0.90,7320,0.582020,0.623295,0.441308,0.461032,...,0.739716,0.575397,-1.253440,0.379019,-0.662340,0.499278,-8.881532,0.137333,-23.764786,-0.072161
2998,model_arch100_r0.9_Ld0.7_Lp0.3_seed1896,[100],0.7,0.3,0.90,1896,0.392292,0.696655,0.010496,0.492522,...,-0.355663,0.639261,-1.786822,0.409191,-0.433355,0.459273,-15.173592,-0.154697,-45.088374,-0.533613


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)

SETS_CATEGORY = {
    "ZZx1":  "Train",
    "ZZx2":     "Val",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZxReto":  "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 10  # top modelos

w_val = 0.4
w_train = 0.4
w_test = 0.1

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 10 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
195,model_arch7_r0.9_Ld0.3_Lp0.7_seed4556,[7],0.847661,0.954623,-4.869805,-0.154097
919,model_arch31_r0.9_Ld0.3_Lp0.7_seed1628,[31],0.430334,0.406228,-2.357825,-0.200820
1457,model_arch49_r0.9_Ld0.3_Lp0.7_seed7320,[49],0.632215,0.543183,-3.441016,-0.250839
1600,model_arch54_r0.01_Ld0.3_Lp0.7_seed4556,[54],0.561191,0.588201,-3.408986,-0.251181
2232,model_arch75_r0.01_Ld0.3_Lp0.7_seed7320,[75],0.632023,0.651724,-3.597405,-0.252880
2805,model_arch94_r0.9_Ld0.3_Lp0.7_seed4556,[94],0.394706,0.638953,-3.236283,-0.253361
1396,model_arch47_r0.9_Ld0.3_Lp0.7_seed6942,[47],0.412694,0.587904,-3.152233,-0.257210
2924,model_arch98_r0.01_Ld0.3_Lp0.7_seed1628,[98],0.400398,0.851773,-3.664542,-0.266326
2299,model_arch77_r0.9_Ld0.3_Lp0.7_seed1628,[77],0.688450,0.610938,-3.666263,-0.268880
2803,model_arch94_r0.01_Ld0.3_Lp0.7_seed1896,[94],0.414030,0.676114,-3.425649,-0.275493



📊 MÉTRICAS COMPLETAS - TOP 10 (theta)


,model,Neurons,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZxReto_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
195,model_arch7_r0.9_Ld0.3_Lp0.7_seed4556,[7],0.847661,0.954623,-5.714005,-10.103905,-4.817523,-2.989746,-3.702921,0.569252,-7.329788,0.847661,0.954623,-4.869805,-0.154097
919,model_arch31_r0.9_Ld0.3_Lp0.7_seed1628,[31],0.430334,0.406228,-0.225041,-8.171587,-5.295831,-0.401771,-0.379865,-0.141900,-1.888779,0.430334,0.406228,-2.357825,-0.200820
1457,model_arch49_r0.9_Ld0.3_Lp0.7_seed7320,[49],0.632215,0.543183,-2.731281,-10.730111,-3.040181,-1.340440,-0.321531,0.199796,-6.123364,0.632215,0.543183,-3.441016,-0.250839
1600,model_arch54_r0.01_Ld0.3_Lp0.7_seed4556,[54],0.561191,0.588201,-2.848713,-10.838724,-3.372571,-1.090181,-0.463453,0.026469,-5.275730,0.561191,0.588201,-3.408986,-0.251181
2232,model_arch75_r0.01_Ld0.3_Lp0.7_seed7320,[75],0.632023,0.651724,-4.280865,-9.559783,-1.337276,-1.043747,-0.219034,0.323718,-9.064849,0.632023,0.651724,-3.597405,-0.252880
2805,model_arch94_r0.9_Ld0.3_Lp0.7_seed4556,[94],0.394706,0.638953,-4.077517,-8.640621,-1.785886,-0.418886,-0.555508,-0.059584,-7.115977,0.394706,0.638953,-3.236283,-0.253361
1396,model_arch47_r0.9_Ld0.3_Lp0.7_seed6942,[47],0.412694,0.587904,-2.706323,-9.888234,-2.847838,-0.600198,-0.735829,0.038968,-5.326177,0.412694,0.587904,-3.152233,-0.257210
2924,model_arch98_r0.01_Ld0.3_Lp0.7_seed1628,[98],0.400398,0.851773,-4.072161,-9.203382,-1.401701,-0.522713,-0.846895,-0.145754,-9.459187,0.400398,0.851773,-3.664542,-0.266326
2299,model_arch77_r0.9_Ld0.3_Lp0.7_seed1628,[77],0.688450,0.610938,-5.142790,-9.046124,-0.872555,-1.248881,-0.021737,0.503311,-9.835065,0.688450,0.610938,-3.666263,-0.268880
2803,model_arch94_r0.01_Ld0.3_Lp0.7_seed1896,[94],0.414030,0.676114,-3.572958,-9.789153,-1.984482,-0.701561,-0.566546,-0.074101,-7.290742,0.414030,0.676114,-3.425649,-0.275493


In [6]:
final_table.to_excel("BestModels-1l.xlsx")